# 07 · Superposition and adversarial examples

**Paper:** "Relationship to Adversarial Robustness", pp. 29–30.
**Experiment:** E06 `adversarial`.

**Learning goals**

1. See why interference terms in WᵀW are attack directions (p. 29).
2. Compute the paper's analytic L2 attack by hand on a two-feature model.
3. Measure vulnerability relative to a model without superposition, and relate it to the
   number of features per dimension 1/D* (pp. 29–30).
4. Know where the measure stops being meaningful (tiny budgets at extreme sparsity).

The E06 quick profile takes about 7 minutes on a laptop CPU (measured: 401 s on an Apple M2).

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition.adversarial import (
    analytic_attack,
    attack_directions,
    no_superposition_model,
    vulnerability,
)
from superposition.experiments.runner import run_experiment
from superposition.models import TiedModel
from superposition.runs import list_runs

pd.set_option("display.max_colwidth", 160)


REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    """Your latest complete quick-profile run of ``name`` with its figures, or a fresh one.

    Runs made without figures (``superposition run --no-figures``, or the replication tests)
    are skipped: this notebook shows the run's figures.
    """
    if REUSE:
        for run in reversed(list_runs(name)):
            quick = yaml.safe_load(run.config_path.read_text())["profile"] == "quick"
            if quick and any(run.figures_dir.glob("*.png")):
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["group", "id", "status", "value"]]

## 1. The attack, by hand

Without superposition the end-to-end weights of feature 0 are (WᵀW)₀ = (1, 0, 0, …): the only
way to move output 0 is to move x₀. With superposition they are (1, ε, −ε, ε, …), and every
nonzero entry is another input direction that moves output 0 (p. 29).

For each feature i, the L2 perturbation of size ε that raises output i's pre-activation the
most is δᵢ = ε (WᵀW)ᵢ/‖(WᵀW)ᵢ‖. The paper evaluates every such δᵢ and keeps the most harmful.

Take the smallest model with superposition: two features as an antipodal pair in one
dimension, W = [1, −1], b = 0. **Predict:** on the all-zero input, how much loss does the best
attack cause, compared with ε² for a model that stores each feature in its own dimension?

In [ ]:
pair = TiedModel.from_kind("relu_output", n_features=2, n_hidden=1, n_inst=1)
with torch.no_grad():
    pair.W.copy_(torch.tensor([[[1.0, -1.0]]]))
    pair.b.zero_()
print("attack directions (one row per attacked feature):")
print(attack_directions(pair.W.detach())[0])

x = torch.zeros(1, 1, 2)  # the all-zero input
epsilon = torch.tensor([0.1])
importance = torch.ones(2)
attacked = analytic_attack(pair, x, importance, epsilon)
reference = analytic_attack(no_superposition_model(2, 1, 1), x, importance, epsilon)
print("damage L′ − L:", attacked.damage.item())
print("no-superposition damage:", reference.damage.item())
print("vulnerability:", vulnerability(attacked, reference).item())

The attack on feature 0 moves the input along (1, −1)/√2, which raises output 0 by ε√2: twice
the squared damage of the reference model, whose best move raises one output by exactly ε.
Superposition packs two features into one direction, and the adversary gets the sum of both
weights for free.

## 2. Vulnerability across sparsity (p. 29)

E06 trains ReLU output models over a sparsity sweep and attacks the best model at each
sparsity with a budget of 0.1 × the average input norm. The first setting is the uniform model
of p. 17 (n = 400, m = 30), which is what the p. 29 axes show; the second is n = 20, m = 5.

**Predict:** in the uniform model, 1/D* goes from 1 (dense) to roughly 4–5 at
1/(1 − S) = 10. How much more vulnerable than the reference do you expect the sparsest model
to be?

In [ ]:
run = quick_run("adversarial")
criteria(run)

In [ ]:
display(Image(filename=str(run.figures_dir / "vulnerability.png")))
display(Image(filename=str(run.figures_dir / "vulnerability_vs_features_per_dimension.png")))

In [ ]:
table = run.read_table("vulnerability")
table[
    [
        "setting",
        "inverse_density",
        "features_per_dimension",
        "vulnerability",
        "epsilon",
        "most_attacked_feature",
        "most_attacked_share",
    ]
].round(3)

In the uniform model, vulnerability tracks features per dimension closely (the paper's p. 30
scatter): flat while there is no superposition, a shoulder while features are antipodal pairs
(1/D* ≈ 2), and a second rise once they pack more densely. The more features share each
dimension, the more interference terms an attacker can combine.

The n = 20, m = 5 model hardly enters superposition over the same range (1/D* stays below
about 2), and its vulnerability rises much less. Look at its row in the acceptance table: this
is where E06 is only a PARTIAL reproduction, and docs/REPRODUCTION.md discusses why.

**The extended n = 20 sweep** (not graded) goes far past the paper's range. There the budget
0.1 × E‖x‖ becomes tiny, because almost every input is zero. Compare it with the biases of the
represented features:

In [ ]:
from superposition.experiments.common import GridResult

grid = GridResult.from_checkpoint(run.load_checkpoint("n20_m5_extended"))
extended = table[table["setting"] == "n20_m5_extended"].reset_index(drop=True)
rows = []
for cond in range(grid.n_conditions):
    W, b, _ = grid.best_weights(cond)
    represented = W.norm(dim=0) > 0.5
    rows.append(
        {
            "1/(1 − S)": round(1 / grid.condition_probability(cond), 1),
            "ε": round(float(extended.loc[cond, "epsilon"]), 4),
            "median bias (represented)": round(float(b[represented].median()), 3),
            "vulnerability": round(float(extended.loc[cond, "vulnerability"]), 2),
            "1/D*": round(float(extended.loc[cond, "features_per_dimension"]), 2),
        }
    )
pd.DataFrame(rows)

Once the biases turn clearly negative and ε is much smaller than them, a push of size ε cannot
lift an output past its threshold. Vulnerability falls back even though 1/D* keeps rising: the
measure only tracks superposition while the budget is large compared with the biases.

## Exercises

**1.** Show that for the no-superposition reference, L′₀ − L₀ = maxᵢ Iᵢ · ε² exactly, for any
input. Check it numerically with `no_superposition_model(6, 3, 1)` and importances 0.9ⁱ.

<details><summary>Solution</summary>

The reference has W = [I₃ | 0], so (WᵀW)ᵢ = eᵢ for i < 3 and 0 otherwise. Attack i adds ε to
input i only, which adds ε to output i (its pre-activation xᵢ + ε is positive, so the ReLU
passes it). The loss grows by Iᵢ ε², and the worst case is the most important represented
feature.

```python
imp = 0.9 ** torch.arange(6.0)
x = torch.rand(1000, 1, 6)
eps = torch.tensor([0.2])
print(analytic_attack(no_superposition_model(6, 3, 1), x, imp, eps).damage, imp[0] * 0.04)
```
</details>

**2.** In the uniform setting, which feature does the worst-case attack target most often? Is
it the feature with the largest ‖(WᵀW)ᵢ‖?

<details><summary>Solution</summary>

```python
from superposition.experiments.common import GridResult

grid = GridResult.from_checkpoint(run.load_checkpoint("uniform_n400_m30"))
rows = table[table["setting"] == "uniform_n400_m30"].reset_index(drop=True)
for cond in range(grid.n_conditions):
    W, _, _ = grid.best_weights(cond)
    norms = (W.T @ W).norm(dim=1)
    target = int(rows.loc[cond, "most_attacked_feature"])
    rank = int((norms > norms[target]).sum())
    print(f"{rows.loc[cond, 'inverse_density']:.2f}: feature {target}, "
          f"chosen for {rows.loc[cond, 'most_attacked_share']:.1%} of inputs, row-norm rank {rank}")
```

Not in general. The worst case is chosen per input, and the choice is spread over many
features: the most frequent target is picked for only a few percent of inputs in the sparse
models. Because each attack direction is normalized to length ε, even a feature the model
barely represents defines a full-size perturbation through the represented features' directions,
so a small row of WᵀW does not make a harmless attack. At 1/(1 − S) = 10 the most frequent
target does have one of the largest rows.
</details>

**3.** Why did the paper use analytic attacks instead of gradient descent (p. 29)?

<details><summary>Solution</summary>

In very sparse regimes the output ReLUs are off for almost every input, so the gradient of the
loss with respect to the input is zero almost everywhere: gradient masking. The analytic attack
does not need a gradient; it pushes along a direction chosen from the weights.
</details>

**4.** Test the budget hypothesis: if the fall of vulnerability in the extended n = 20 sweep
comes from ε being small next to the models' negative biases, a larger budget should delay
it. Re-run only that sweep with `budget_fraction: 0.5` and compare.

<details><summary>Solution</summary>

```python
bigger = run_experiment(
    "adversarial", "quick", device="auto",
    overrides={
        "settings": [{"label": "n20_m5_extended", "n_features": 20, "n_hidden": 5,
                      "importance_decay": 0.7, "inverse_density_min": 10.0,
                      "inverse_density_max": 1000.0, "n_points": 7, "n_restarts": 4,
                      "batch_size": None, "attack_samples": 100000, "graded": False}],
        "thresholds": {"budget_fraction": 0.5},
    },
)
bigger.read_table("vulnerability")[["inverse_density", "vulnerability", "features_per_dimension"]]
```

Compare the `vulnerability` column with the extended rows of the cited run (`table` above).
If the hypothesis holds, vulnerability stays above 1 further into the sparse regime; if it
does not, look at `most_attacked_share` and the biases for another explanation. Either way,
write down what you expected before you look.
</details>